In [1]:
def get_repetition_penalty_reward(ngram_size: int = 3, max_penalty: float = -0.1):
    """
    返回一个重复惩罚奖励函数。对生成文本中 n - 元组的重复情况进行惩罚。
    """
    # 检查最大惩罚值是否为正数，如果是则抛出错误
    if max_penalty > 0:
        raise ValueError(f"max_penalty {max_penalty} 不应该为正数")

    def zipngram(text: str, ngram_size: int):
        """辅助函数，用于从文本中生成 n - 元组。"""
        # 将文本转换为小写并按单词分割
        words = text.lower().split()
        # 通过 zip 函数生成 n - 元组
        return zip(*[words[i:] for i in range(ngram_size)])

    def repetition_penalty_reward(completions, **kwargs):
        """
        重复惩罚奖励函数。
        """
        # 从每个完成项中提取内容
        contents = [completion[0]["content"] for completion in completions]
        # 用于存储每个完成项的奖励值
        rewards = []
        for completion in contents:
            # 如果完成项为空，不进行惩罚，奖励值为 0
            if completion == "":
                rewards.append(0.0)
                continue
            # 如果完成项的单词数量少于 n - 元组的大小，不进行惩罚，奖励值为 0
            if len(completion.split()) < ngram_size:
                rewards.append(0.0)
                continue

            # 使用集合来存储唯一的 n - 元组
            ngrams = set()
            # 用于统计总的 n - 元组数量
            total = 0
            # 生成完成项的 n - 元组
            for ng in zipngram(completion, ngram_size):
                # 将 n - 元组添加到集合中，集合会自动忽略重复项
                ngrams.add(ng)
                # 总 n - 元组数量加 1
                total += 1

            # 计算缩放因子，重复越多，缩放因子越大
            scaling = 1 - len(ngrams) / total
            # 根据缩放因子应用惩罚，得到奖励值
            reward = scaling * max_penalty
            # 将奖励值添加到奖励列表中
            rewards.append(reward)
        return rewards

    # 返回重复惩罚奖励函数
    return repetition_penalty_reward